# 02 · Experiments

3 representations × 3 architectures, each evaluated with k-fold cross-validation.
Every run saves its fold scores to `results/<architecture>__<representation>.json`.

In [1]:
# Setup: make the course code (part1) and our code (src) importable.
# Run notebooks from this folder (JupyterLab / VS Code do this by default).
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

%load_ext autoreload
%autoreload 2

In [2]:
from part1 import experiment
from src import config, data, encoders, results_io

## Data and folds

In [3]:
LABEL_MODE = "fine"   # "fine" (28) | "ekman" (7) | "sentiment" (4)
pairs, outcome_space, stats = data.load_pairs(label_mode=LABEL_MODE)
print(stats)
folds = experiment.cross_validation_split(pairs, k=config.K_FOLDS, seed=config.SEED)

{'label_mode': 'fine', 'min_raters': 2, 'comments_with_label': 54263, 'kept': 45446, 'dropped_multilabel': 8817}


## Tokeniser

Reuse the tutorial's or train our own? On what corpus, which vocab size?
Note the choice + reason in the decisions log.

In [4]:
corpus = data.tokenizer_corpus()
tokenizer = experiment.train_tokenizer(corpus, vocab_size=10_000, min_frequency=2)  # TODO: tune
tokenizer.get_vocab_size()

10000

## Representations

In [5]:
representations = {
    "indicator": encoders.TokenIndicator(tokenizer),
    "bow": encoders.BagOfWords(tokenizer),
    # "ours": encoders.ThirdRepresentation(tokenizer),
}

## Log-linear

In [6]:
for rep_name, enc in representations.items():
    name = f"loglin__{rep_name}"
    out = experiment.cross_validation_experiment(
        folds, outcome_space, enc, "logistic", save_to=str(config.MODELS_DIR / name)
    )
    results_io.save_cv(name, out, vocab_size=tokenizer.get_vocab_size())
    print(name, out["accuracies"])

  0%|          | 0/5 [00:00<?, ?it/s]

/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: T

loglin__indicator OrderedDict({0: 0.5295929592959296, 1: 0.5457146000660139, 2: 0.5320717350643636, 3: 0.5384530751457807, 4: 0.5310815271206953})


  0%|          | 0/5 [00:00<?, ?it/s]

/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: T

loglin__bow OrderedDict({0: 0.5004400440044005, 1: 0.5321817581692155, 2: 0.506106282319287, 3: 0.5103971834085158, 4: 0.5045659588513588})


## MLP (one hidden layer)

In [7]:
smlp_args = dict(hidden_dim=128, dropout=0.1, num_epochs=10)  # TODO: tune

for rep_name, enc in representations.items():
    name = f"mlp__{rep_name}"
    out = experiment.cross_validation_experiment(
        folds, outcome_space, enc, "mlp", save_to=str(config.MODELS_DIR / name), smlp_args=smlp_args
    )
    results_io.save_cv(name, out, vocab_size=tokenizer.get_vocab_size(), **smlp_args)
    print(name, out["accuracies"])

  0%|          | 0/5 [00:00<?, ?it/s]

Transforming documents...
Document transformation done! Time taken: 0.878 seconds
Dataloader done! Time taken: 1.249 seconds
Training...


  0%|          | 0/10 [00:00<?, ?it/s]

/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/part1/smlp.py:68: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:252.)
  indices = torch.tensor([x.row, x.col], dtype=torch.long)
/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/part1/smlp.py:71: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:923.)
  return torch.sparse_coo_tensor(indices, values, size=x.shape).coalesce()

Epoch 01 | train loss: 2.2480 | train acc: 0.4327 | val loss: 1.8258 | val acc: 0.5077
Epoch 02 | train loss: 1.3915 | train acc: 0.6010 | val loss: 1.6460 | val acc: 0.5422
Epoch 03 | train loss: 0.9614 | train acc: 0.7117 | val loss: 1.6908 | val acc: 0.5417
Epoch 04 | train loss: 0.6885 | train acc: 0.7941 | val loss: 1.8106 | val acc: 0.5303
Epoch 05 | train loss: 0.5069 | train acc: 0.8528 | val loss: 1.9811 | val acc: 0.5283
Epoch 06 | train loss: 0.3820 | train acc: 0.8924 | val loss: 2.1557 | val acc: 0.5127
Epoch 07 | train loss: 0.2908 | train acc: 0.9224 | val loss: 2.3526 | val acc: 0.5109
Epoch 08 | train loss: 0.2256 | train acc: 0.9416 | val loss: 2.5497 | val acc: 0.5053
Epoch 09 | train loss: 0.1779 | train acc: 0.9560 | val loss: 2.7094 | val acc: 0.4979
Epoch 10 | train loss: 0.1428 | train acc: 0.9664 | val loss: 2.8839 | val acc: 0.4921
Training done! Time taken: 18.151 seconds
Transforming documents...
Document transformation done! Time taken: 0.965 seconds
Datalo

  0%|          | 0/10 [00:00<?, ?it/s]

... (15 outputs left out)

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 2.2534 | train acc: 0.4328 | val loss: 1.8066 | val acc: 0.5145
Epoch 02 | train loss: 1.4047 | train acc: 0.5954 | val loss: 1.6178 | val acc: 0.5487
Epoch 03 | train loss: 0.9658 | train acc: 0.7108 | val loss: 1.6600 | val acc: 0.5495
Epoch 04 | train loss: 0.6904 | train acc: 0.7915 | val loss: 1.7813 | val acc: 0.5421
Epoch 05 | train loss: 0.5074 | train acc: 0.8495 | val loss: 1.9432 | val acc: 0.5341
Epoch 06 | train loss: 0.3779 | train acc: 0.8940 | val loss: 2.1073 | val acc: 0.5204
Epoch 07 | train loss: 0.2895 | train acc: 0.9226 | val loss: 2.2984 | val acc: 0.5138
Epoch 08 | train loss: 0.2207 | train acc: 0.9437 | val loss: 2.4997 | val acc: 0.5114
Epoch 09 | train loss: 0.1710 | train acc: 0.9597 | val loss: 2.6704 | val acc: 0.5043
Epoch 10 | train loss: 0.1374 | train acc: 0.9687 | val loss: 2.8629 | val acc: 0.5002
Training done! Time taken: 16.148 seconds
Transforming documents...
Document transformation done! Time taken: 1.027 seconds
Datalo

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 2.2381 | train acc: 0.4384 | val loss: 1.8184 | val acc: 0.5101
Epoch 02 | train loss: 1.3975 | train acc: 0.5988 | val loss: 1.6365 | val acc: 0.5432
Epoch 03 | train loss: 0.9637 | train acc: 0.7106 | val loss: 1.6844 | val acc: 0.5415
Epoch 04 | train loss: 0.6892 | train acc: 0.7926 | val loss: 1.7982 | val acc: 0.5233
Epoch 05 | train loss: 0.5035 | train acc: 0.8530 | val loss: 1.9790 | val acc: 0.5206
Epoch 06 | train loss: 0.3766 | train acc: 0.8936 | val loss: 2.1526 | val acc: 0.5128
Epoch 07 | train loss: 0.2850 | train acc: 0.9243 | val loss: 2.3361 | val acc: 0.5028
Epoch 08 | train loss: 0.2200 | train acc: 0.9449 | val loss: 2.5145 | val acc: 0.5005
Epoch 09 | train loss: 0.1723 | train acc: 0.9587 | val loss: 2.6940 | val acc: 0.4870
Epoch 10 | train loss: 0.1375 | train acc: 0.9683 | val loss: 2.9006 | val acc: 0.4884
Training done! Time taken: 16.058 seconds
mlp__bow OrderedDict({0: 0.5398239823982398, 1: 0.5644185278908571, 2: 0.548575200792166

## Our third architecture: deep low-rank MLP

`src/models.py::DeepLowRankMLP`: several hidden layers, each weight matrix factorised as W = A·B with rank r
(the LoRA factorisation, trained from scratch: there is no pretrained model to adapt).
The first layer (vocab × hidden) holds most parameters; the rank bottleneck shrinks it so we can stack more layers.
Residual connections between equal-width layers keep the deep network trainable.

In [8]:
from src import models

# Quick check before the full CV run: parameter counts of full-rank vs low-rank
V, C = tokenizer.get_vocab_size(), len(outcome_space)
for r in [None, 64, 16]:
    m = models.DeepLowRankMLP(V, [256, 256, 256, 256], C, rank=r)
    print(f"rank={r}: {models.count_parameters(m):,} parameters")

rank=None: 15,257,628 parameters
rank=64: 3,886,108 parameters
rank=16: 977,692 parameters


In [19]:
deep_args = dict(hidden_dims=[256, 256, 256, 256], rank=32, dropout=0.5, residual=True)  # TODO: tune
deep_train_args = dict(num_epochs=10, learning_rate=1e-3, batch_size=64)

for rep_name, enc in representations.items():
    name = f"deeplowrank__{rep_name}"
    out = models.cross_validation_experiment(
        folds, outcome_space, enc, save_to=str(config.MODELS_DIR / name),
        model_args=deep_args, train_args=deep_train_args,
    )
    results_io.save_cv(name, out, vocab_size=tokenizer.get_vocab_size(), **deep_args, **deep_train_args)
    print(name, out["accuracies"])

  0%|          | 0/5 [00:00<?, ?it/s]

DeepLowRankMLP: 1,947,164 trainable parameters (encoding took 11.2s)


  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss 2.2087 acc 0.4425 | val loss 1.9262 acc 0.4926
Epoch 02 | train loss 1.4535 acc 0.5961 | val loss 1.8416 acc 0.5184
Epoch 03 | train loss 0.8450 acc 0.7483 | val loss 2.1414 acc 0.4770
Epoch 04 | train loss 0.4971 acc 0.8520 | val loss 2.6100 acc 0.4525
Epoch 05 | train loss 0.3181 acc 0.9053 | val loss 3.2025 acc 0.4705
Epoch 06 | train loss 0.2319 acc 0.9290 | val loss 3.7811 acc 0.4614
Epoch 07 | train loss 0.1811 acc 0.9445 | val loss 3.8723 acc 0.4410
Epoch 08 | train loss 0.1458 acc 0.9547 | val loss 4.5920 acc 0.4482
Epoch 09 | train loss 0.1249 acc 0.9611 | val loss 4.5528 acc 0.4417
Epoch 10 | train loss 0.1044 acc 0.9674 | val loss 4.6601 acc 0.4308
DeepLowRankMLP: 1,947,164 trainable parameters (encoding took 11.1s)


  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss 2.2113 acc 0.4422 | val loss 1.8868 acc 0.5109
Epoch 02 | train loss 1.4794 acc 0.5857 | val loss 1.8663 acc 0.5200
Epoch 03 | train loss 0.8829 acc 0.7416 | val loss 2.0658 acc 0.4863
Epoch 04 | train loss 0.5269 acc 0.8449 | val loss 2.5166 acc 0.4815
Epoch 05 | train loss 0.3414 acc 0.8978 | val loss 2.9738 acc 0.4646
Epoch 06 | train loss 0.2411 acc 0.9275 | val loss 3.2602 acc 0.4653
Epoch 07 | train loss 0.1901 acc 0.9420 | val loss 3.7897 acc 0.4542
Epoch 08 | train loss 0.1545 acc 0.9527 | val loss 4.2984 acc 0.4650
Epoch 09 | train loss 0.1349 acc 0.9590 | val loss 4.1658 acc 0.4501
Epoch 10 | train loss 0.1087 acc 0.9661 | val loss 4.3989 acc 0.4549
DeepLowRankMLP: 1,947,164 trainable parameters (encoding took 11.3s)


  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss 2.2100 acc 0.4419 | val loss 1.9612 acc 0.4921
Epoch 02 | train loss 1.4749 acc 0.5872 | val loss 1.8908 acc 0.5170
Epoch 03 | train loss 0.8728 acc 0.7434 | val loss 2.2463 acc 0.4743
Epoch 04 | train loss 0.5188 acc 0.8452 | val loss 2.8544 acc 0.4695
Epoch 05 | train loss 0.3314 acc 0.9009 | val loss 3.1750 acc 0.4524
Epoch 06 | train loss 0.2297 acc 0.9318 | val loss 3.9893 acc 0.4444
Epoch 07 | train loss 0.1860 acc 0.9447 | val loss 3.8087 acc 0.4405


KeyboardInterrupt: 